# Part F — Eigenvalues & Eigenvectors

**Final Project: Philippine Peso Depreciation Early-Warning Model: Predicting the Next-Month PHP/USD Exchange Rate Using Linear Regression and PCA**

Eigen-decomposition reveals the directions along which your data actually varies — information that is invisible in the raw feature list.


Student name:* Alfajaro, Ghan Vincent | Arile, Charles Jullianne | Asuar, Jhoe Maven | Mondragon, Leneth | Ulitin, Justin |   *Date:* November 19, 2026

---


**What to do:** Compute the eigenvalues and eigenvectors of a matrix relevant to your data (e.g., the covariance matrix of your standardized training features).

In [1]:
# Run this first: recall variables saved from the previous part's notebook.
# (If this is your first notebook in the session, or the store is empty,
#  it fetches directly from the official BSP URL and applies Part D's exact transformation.)
%store -r X_train X_test y_train y_test feature_names

import numpy as np
import pandas as pd

try:
    _ = (X_train, X_test, y_train, y_test, feature_names)
    print("Loaded X_train, X_test, y_train, y_test, feature_names from %store.")
except NameError:
    print("Store is empty: loading from BSP URL and rebuilding Part D chronological split...")
    BSP_URL = "https://www.bsp.gov.ph/statistics/external/pesodollar.xlsx"
    monthly_raw = pd.read_excel(BSP_URL, sheet_name="monthly", header=4, usecols="A:D")

    year = pd.to_numeric(monthly_raw["Period"], errors="coerce").ffill()
    month_name = monthly_raw["Unnamed: 2"].astype(str).str.strip().str.lower()
    rate = pd.to_numeric(monthly_raw["Average"], errors="coerce")

    month_numbers = {
        "january": 1, "february": 2, "march": 3, "april": 4,
        "may": 5, "june": 6, "july": 7, "august": 8,
        "september": 9, "october": 10, "november": 11, "december": 12
    }
    dates = pd.to_datetime({"year": year, "month": month_name.map(month_numbers), "day": 1}, errors="coerce")

    monthly = pd.DataFrame({"Date": dates, "Rate": rate}).dropna()
    monthly["Date"] = pd.to_datetime(monthly["Date"]).dt.to_period("M").dt.to_timestamp()
    monthly = monthly.sort_values("Date").drop_duplicates("Date", keep="last").set_index("Date")["Rate"].astype(float)

    expected_months = pd.date_range(start="1990-01-01", end="2026-09-01", freq="MS")
    monthly = monthly.loc["1990-01-01":"2026-09-01"].reindex(expected_months)

    model_data = pd.DataFrame(index=monthly.index)
    model_data["CurrentRate"] = monthly
    model_data["Trailing3MonthAverage"] = monthly.rolling(window=3).mean()
    model_data["YearOverYearChange"] = monthly - monthly.shift(12)
    model_data["NextMonthRate"] = monthly.shift(-1)

    feature_names = ["CurrentRate", "Trailing3MonthAverage", "YearOverYearChange"]
    model_data = model_data.dropna(subset=feature_names + ["NextMonthRate"])

    X = model_data[feature_names].to_numpy(dtype=float)
    y = model_data["NextMonthRate"].to_numpy(dtype=float)

    # Exact chronological 80/20 split and training-set standardization from Part D
    train_size = int(0.8 * X.shape[0])
    X_train_raw = X[:train_size]
    mu = X_train_raw.mean(axis=0)
    sigma = X_train_raw.std(axis=0, ddof=0)
    S = np.diag(1.0 / sigma)

    X_transformed = (X - mu) @ S
    X_train, X_test = X_transformed[:train_size], X_transformed[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]
    print(f"Rebuilt X_train {X_train.shape} and X_test {X_test.shape} from BSP URL.")

Loaded X_train, X_test, y_train, y_test, feature_names from %store.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

In [3]:
# TODO: Compute the covariance matrix C of your (standardized) training features.
Z = (X_train - X_train.mean(axis=0)) / X_train.std(axis=0, ddof=1)

C = np.cov(Z, rowvar=False)
print("C shape:", C.shape)
print("Symmetric?", np.allclose(C, C.T))
print(C)

C shape: (3, 3)
Symmetric? True
[[1.     0.9979 0.1049]
 [0.9979 1.     0.0767]
 [0.1049 0.0767 1.    ]]


In [4]:
eigvals, eigvecs = np.linalg.eigh(C)

idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

print("Eigenvalues:", eigvals)
print("Eigenvectors (each column is one eigenvector):")
print(eigvecs)

Eigenvalues: [2.0141 0.9841 0.0017]
Eigenvectors (each column is one eigenvector):
[[-0.7024 -0.0746 -0.7079]
 [-0.7006 -0.1031  0.706 ]
 [-0.1256  0.9919  0.0201]]


In [5]:
# TODO: Verify C v = lambda v for the leading eigenpair
# v1, lam1 = eigvecs[:, 0], eigvals[0]
# error = np.max(np.abs(C @ v1 - lam1 * v1))
# print("max|C v1 - lambda1 v1| =", error)
v1, lam1 = eigvecs[:, 0], eigvals[0]
error = np.max(np.abs(C @ v1 - lam1 * v1))
print("lambda1 =", lam1)
print("v1 =", v1)
print("C v1 =", C @ v1)
print("lambda1 * v1 =", lam1 * v1)
print("max|C v1 - lambda1 v1| =", error)

lambda1 = 2.014146292115373
v1 = [-0.7024 -0.7006 -0.1256]
C v1 = [-1.4147 -1.4112 -0.253 ]
lambda1 * v1 = [-1.4147 -1.4112 -0.253 ]
max|C v1 - lambda1 v1| = 6.661338147750939e-16


In [6]:
# TODO: Compute the explained variance ratio for each component and
# inspect which original features load most heavily on the first eigenvector.
ratio = eigvals / eigvals.sum()
print("Explained variance ratio:", ratio)
print("Percent:", ratio * 100)

names = ['CurrentRate', 'Trailing3MonthAverage', 'YearOverYearChange']
print("\nFirst eigenvector loadings:")
for n, w in zip(names, eigvecs[:, 0]):
    print(n, ":", round(w, 4))


Explained variance ratio: [0.6714 0.328  0.0006]
Percent: [67.1382 32.8049  0.0569]

First eigenvector loadings:
CurrentRate : -0.7024
Trailing3MonthAverage : -0.7006
YearOverYearChange : -0.1256


### Answer these in your report

**Q21.** Which matrix did you decompose, and why is it symmetric? Why does symmetry matter for this decomposition?

> *Your answer:* _(I decomposed the covariance matrix C of my standardized training features. It is symmetric because the covariance between feature i and feature j is the same as between j and i, so C equals its transpose. Symmetry matters because a symmetric matrix always has real eigenvalues and eigenvectors that are perpendicular to each other, which makes the decomposition clean and lets me use `np.linalg.eigh`.)_

**Q22.** List your eigenvalues from largest to smallest. What percentage of total variance does the single largest eigenvalue explain?

> *Your answer:* _(My eigenvalues from largest to smallest are 2.0141, 0.9841, 0.0017. The largest one explains about 67.1% of the total variance.)_

**Q23.** Verify numerically that Cv = λv for at least one eigenpair. Report the vectors/values you used and the resulting numerical error.

> *Your answer:* _(I used the largest eigenvalue λ1 = 2.0141 and its eigenvector v1 = [-0.7024, -0.7006,-0.12561]. Computing C v1 and λ1 v1 gave the same vector, and the biggest difference was about 6.661338147750939e-16. That is basically zero, just rounding error from the computer, so Cv = λv holds.)_

**Q24.** Look at the eigenvector associated with the largest eigenvalue. Which original features contribute most strongly to it, and what does that tell you about your data?

> *Your answer:* _(CurrentRate and Trailing3MonthAverage have the biggest loadings in the first eigenvector and they are about the same size. The YearOverYearChange loading is much smaller. This tells me the current rate and the 3-month average move together almost exactly, so they carry nearly the same information. Most of the variation in my data is just the level of the peso-dollar rate.)_

### Save your work for the next notebook

In [7]:
%store C
%store eigvals
%store eigvecs
print("Saved C, eigvals, eigvecs for Part G.")

Stored 'C' (ndarray)
Stored 'eigvals' (ndarray)
Stored 'eigvecs' (ndarray)
Saved C, eigvals, eigvecs for Part G.
